In [30]:
import pandas as pd

movies = pd.read_csv("../datasets/raw/movies.csv")

# Check for duplicates
# print("Duplicate rows:", movies.duplicated().sum())
# print("Duplicate movie IDs:", movies["movieId"].duplicated().sum())
# print("Duplicate titles:", movies["title"].duplicated().sum())
# Check the genres
# print("Number of unique genre combinations:", movies["genres"].nunique())

# print("\nSample genres:")
# print(movies["genres"].head(20).to_string(index=False))
genres = movies["genres"].str.split("|").explode()

# print("\nUnique genres:")
# print(genres.unique())

# print("\nGenre count:")
# print(genres.value_counts())
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")
# print(movies[["title", "year"]].head(10))
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")
# Clean the Movie Title
movies["clean_title"] = movies["title"].str.replace(
    r"\s*\(\d{4}\)\s*$",
    "",
    regex=True
)
# print(movies[["title", "clean_title", "year"]].head(10))
# Handle Missing/Unknown Genres
# print(
#     movies["genres"].value_counts().tail(10)
# )
unknown_genres = movies[
    movies["genres"] == "(no genres listed)"
]

# print("Movies without genres:", len(unknown_genres))
movies["genres"] = movies["genres"].replace(
    "(no genres listed)",
    ""
)
# Create a Genre List
movies["genre_list"] = movies["genres"].apply(
    lambda x: x.split("|") if x else []
)
# print(
#     movies[["clean_title", "genre_list"]].head(10)
# )

# Create Genre Text
movies["genre_text"] = movies["genre_list"].apply(
    lambda genres: " ".join(genres)
)
# Basic Genre Analysis
genre_counts = (
    movies["genres"]
    .str.split("|")
    .explode()
    .value_counts()
)

# print(genre_counts)
# print(genre_counts.head(10))
# Check Year Distribution
# print(
#     movies["year"].value_counts().sort_index().tail(20)
# )
# Processed Dataset
movies_processed = movies[
    [
        "movieId",
        "title",
        "clean_title",
        "year",
        "genres",
        "genre_list",
        "genre_text"
    ]
].copy()
movies_processed.to_csv(
    "../datasets/processed/movies_processed.csv",
    index=False
)
# print(movies_processed.shape)
# print(movies_processed.head())

In [53]:
# Process ratings.csv
ratings_sample = pd.read_csv("../datasets/raw/ratings.csv", nrows=100000)

# print(ratings_sample.head())
# print(ratings_sample.shape)
# print(ratings_sample.info())
# print(ratings_sample["rating"].describe())
# print(
#     ratings_sample["rating"].value_counts().sort_index()
# )

# Check data quality
# print("Missing values:")
# print(ratings_sample.isnull().sum())

# print("\nDuplicate rows:")
# print(ratings_sample.duplicated().sum())
# print(
#     "Unique users:",
#     ratings_sample["userId"].nunique()
# )

# print(
#     "Unique movies:",
#     ratings_sample["movieId"].nunique()
# )

# Process the complete ratings file in chunks
ratings_file = "../datasets/raw/ratings.csv"

chunk_size = 500_000

total_ratings = 0
unique_users = set()
unique_movies = set()

# rating_counts = {}

# for chunk in pd.read_csv(ratings_file, chunksize=chunk_size):
#     total_ratings += len(chunk)

#     unique_users.update(chunk["userId"].unique())

#     unique_movies.update(chunk["movieId"].unique())

#     counts = chunk["rating"].value_counts()

#     for rating, count in counts.items():
#         rating_counts[rating] = rating_counts.get(rating, 0) + count

# print("Total ratings:", total_ratings)
# print("Unique users:", len(unique_users))
# print("Unique movies:", len(unique_movies))
# print("Rating distribution:", rating_counts)

movie_stats = {}

for chunk in pd.read_csv(ratings_file, chunksize=chunk_size):
    grouped = chunk.groupby("movieId")["rating"].agg(["count", "sum"])
    for movie_id, row in grouped.iterrows():

        if movie_id not in movie_stats:
            movie_stats[movie_id] = {"rating_count": 0, "rating_sum": 0}

        movie_stats[movie_id]["rating_count"] += int(row["count"])

        movie_stats[movie_id]["rating_sum"] += float(row["sum"])

movie_stats_df = pd.DataFrame.from_dict(
    movie_stats,
    orient="index"
)

movie_stats_df.index.name = "movieId"

movie_stats_df["average_rating"] = (
    movie_stats_df["rating_sum"] /
    movie_stats_df["rating_count"]
)

movie_stats_df.reset_index(inplace=True)
# print(movie_stats_df.head())
# Find the Most Rated Movies
movie_stats_df.sort_values(
    "rating_count",
    ascending=False
).head(20)
# Find Highest Rated Movies
movie_stats_df[
    movie_stats_df["rating_count"] >= 100
].sort_values(
    "average_rating",
    ascending=False
).head(20)
# Merge Movie Statistics
movies_final = movies_processed.merge(
    movie_stats_df,
    on="movieId",
    how="left"
)
# print(movies_final.head())
movies_final.to_csv(
    "../datasets/processed/movies_final.csv",
    index=False
)

In [ ]:
rating_distribution = (
    ratings_sample["rating"]
    .value_counts()
    .sort_index()
)

# print(rating_distribution)
import matplotlib.pyplot as plt

rating_distribution.plot(kind="bar")

plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.show()
print("Unique users in sample:",
      ratings_sample["userId"].nunique())

print("Unique movies in sample:",
      ratings_sample["movieId"].nunique())

print("Duplicate rows:",
      ratings_sample.duplicated().sum())

print("\nMissing values:")
print(ratings_sample.isnull().sum())

In [ ]:
print(movies.columns.tolist())
print(movies.head())

In [35]:
tags = pd.read_csv("../datasets/raw/tags.csv")

# print(tags.head())
# print(tags.shape)
# print(tags.info())

# print("Missing values:")
# print(tags.isnull().sum())

# print("\nDuplicate rows:")
# print(tags.duplicated().sum())

# print("\nUnique movies with tags:")
# print(tags["movieId"].nunique())

# print("\nUnique tags:")
# print(tags["tag"].nunique())

# print("\nNumber of users creating tags:")
# print(tags["userId"].nunique())
# normalize
tags["tag"] = (
    tags["tag"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
# remove empty tags
tags = tags[tags["tag"] != ""].copy()
# Aggregate tags by movie and remove duplicate tags within each movie
# movieId   tag_text
# 296       action dark funny thriller
movie_tags = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.drop_duplicates()))
    .reset_index(name="tag_text")
)
# Remove previously created tag columns if they already exist
movies = movies.drop(
    columns=["tag_text", "content_text"],
    errors="ignore"
)

# Merge tags with movies
movies = movies.merge(
    movie_tags,
    on="movieId",
    how="left"
)

# print(movies.columns.tolist())
# print(
#     "Movies without tags:",
#     movies["tag_text"].isna().sum()
# )
# Replace missing tags with an empty string
movies["tag_text"] = movies["tag_text"].fillna("")
# combine genres and tags:
movies["content_text"] = (
    movies["genre_text"] + " " +
    movies["tag_text"]
).str.strip()

print(movies.columns.tolist())


['movieId', 'title', 'genres', 'year', 'clean_title', 'genre_list', 'genre_text', 'genome_tag_text', 'tag_text', 'content_text']


In [ ]:
# print(movies[
#     [
#         "movieId",
#         "clean_title",
#         "genre_text",
#         "tag_text",
#         "content_text"
#     ]
# ].head(10))
print(movies.shape)
print(movies.columns.tolist())
print(movies[["movieId", "clean_title", "tag_text"]].head())

In [ ]:
genome_tags = pd.read_csv(
    "../datasets/raw/genome-tags.csv"
)

print(genome_tags.head())
print(genome_tags.shape)
print(genome_tags.info())
# print(genome_tags.isnull().sum())
# print("Unique tags:", genome_tags["tag"].nunique())
# print("Unique tag IDs:", genome_tags["tagId"].nunique())

genome_scores_sample = pd.read_csv(
    "../datasets/raw/genome-scores.csv",
    nrows=100000
)

print(genome_scores_sample.head())
print(genome_scores_sample.shape)
print(genome_scores_sample.info())

print(genome_scores_sample["relevance"].describe())

# print("Genome tags:")
# print(genome_tags.head(10))

# print("\nGenome scores:")
# print(genome_scores_sample.head(10))
# genome_scores_sample.merge(
#     genome_tags,
#     on="tagId",
#     how="left"
# ).head(20)

In [ ]:
print(genome_scores_sample["relevance"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

# Count strong relationships

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.8]

for threshold in thresholds:
    count = (
        genome_scores_sample["relevance"] >= threshold
    ).sum()

    percentage = count / len(genome_scores_sample) * 100

    print(
        f"Threshold {threshold}: "
        f"{count:,} rows ({percentage:.2f}%)"
    )

In [ ]:

from collections import defaultdict

genome_scores_file = "../datasets/raw/genome-scores.csv"

THRESHOLD = 0.3
CHUNK_SIZE = 500_000

# movieId -> set of tagIds
movie_genome_tags = defaultdict(set)

chunk_number = 0

for chunk in pd.read_csv(
    genome_scores_file,
    chunksize=CHUNK_SIZE
):
    chunk_number += 1

    # Keep only meaningful tag relationships
    filtered = chunk[
        chunk["relevance"] >= THRESHOLD
    ]

    # Add tag IDs to each movie
    for movie_id, tag_id in zip(
        filtered["movieId"],
        filtered["tagId"]
    ):
        movie_genome_tags[movie_id].add(tag_id)

    print(
        f"Processed chunk {chunk_number}: "
        f"{len(filtered):,} relevant rows"
    )

print("Finished processing genome scores.")
print("Movies with genome tags:", len(movie_genome_tags))

In [ ]:
# Convert Tag IDs to Actual Tag Names

tag_lookup = dict(
    zip(
        genome_tags["tagId"],
        genome_tags["tag"]
    )
)
movie_genome_tag_text = {}

for movie_id, tag_ids in movie_genome_tags.items():

    tag_names = [
        tag_lookup[tag_id]
        for tag_id in tag_ids
        if tag_id in tag_lookup
    ]

    movie_genome_tag_text[movie_id] = " ".join(
        sorted(set(tag_names))
    )
genome_features = pd.DataFrame(
    [
        {
            "movieId": movie_id,
            "genome_tag_text": tag_text
        }
        for movie_id, tag_text
        in movie_genome_tag_text.items()
    ]
)
print(genome_features.head())
print(genome_features.shape)

In [50]:
# movies = movies.drop(
#     columns=["genome_tag_text"],
#     errors="ignore"
# )

movies = movies.drop(
    columns=["genome_tag_text_x", "genome_tag_text_y"],
    errors="ignore"
)


In [ ]:
movies = movies.merge(
    genome_features,
    on="movieId",
    how="left"
)
# print(movies.columns.tolist())

movies["content_text"] = (
    movies["genre_text"] + " " +
    movies["tag_text"] + " " +
    movies["genome_tag_text"]
).str.strip()


# print(
#     movies[
#         [
#             "movieId",
#             "clean_title",
#             "genre_text",
#             "tag_text",
#             "genome_tag_text",
#             "content_text"
#         ]
#     ].head()
# )

movies.to_csv(
    "../datasets/processed/movies_with_genome.csv",
    index=False
)
genome_features.to_csv(
    "../datasets/processed/genome_features.csv",
    index=False
)

In [59]:
# Clean any previous rating columnsClean any previous rating columns
rating_columns = [
    "rating_count",
    "rating_sum",
    "average_rating"
]

movies = movies.drop(
    columns=rating_columns,
    errors="ignore"
)
movies = movies.merge(
    movies_final[
        [
            "movieId",
            "rating_count",
            "rating_sum",
            "average_rating"
        ]
    ],
    on="movieId",
    how="left"
)
# Handle movies without ratings
movies["rating_count"] = (
    movies["rating_count"]
    .fillna(0)
    .astype("int64")
)

movies["rating_sum"] = (
    movies["rating_sum"]
    .fillna(0)
)

movies["average_rating"] = (
    movies["average_rating"]
    .fillna(0)
)
# print(movies.columns.tolist())
print(
    movies[
        [
            "movieId",
            "clean_title",
            "average_rating",
            "rating_count"
        ]
    ].head(10)
)
movies.to_csv(
    "../datasets/processed/movies_final_phase1.csv",
    index=False
)

   movieId                  clean_title  average_rating  rating_count
0        1                    Toy Story        3.893708         57309
1        2                      Jumanji        3.251527         24228
2        3             Grumpier Old Men        3.142028         11804
3        4            Waiting to Exhale        2.853547          2523
4        5  Father of the Bride Part II        3.058434         11714
5        6                         Heat        3.854909         24588
6        7                      Sabrina        3.363666         12132
7        8                 Tom and Huck        3.114583          1344
8        9                 Sudden Death        2.992051          3711
9       10                    GoldenEye        3.421458         28265


In [ ]:
# print("Rows:", len(movies))
# print("Columns:", len(movies.columns))
# print(movies.isnull().sum())

# print("Rows:", movies.shape[0])
# print("Columns:", movies.shape[1])

# print("\nColumns:")
# print(movies.columns.tolist())

# Phase 1 validation

# 1. Duplicate movie IDs
print("Duplicate movie IDs:",
      movies["movieId"].duplicated().sum())

# 2. Missing values
print("\nMissing values:")
print(movies.isnull().sum())

# 3. Invalid average ratings
invalid_ratings = movies[
    (movies["average_rating"] < 0) |
    (movies["average_rating"] > 5)
]

print("\nInvalid average ratings:",
      len(invalid_ratings))

# 4. Invalid rating counts
invalid_counts = movies[
    movies["rating_count"] < 0
]

print("Invalid rating counts:",
      len(invalid_counts))

# 5. Empty content
empty_content = (
    movies["content_text"]
    .fillna("")
    .str.strip()
    .eq("")
)

print("Movies with empty content:",
      empty_content.sum())

# 6. Movies with ratings
print("Movies with ratings:",
      (movies["rating_count"] > 0).sum())

print("Movies without ratings:",
      (movies["rating_count"] == 0).sum())

In [ ]:



text_columns = [
    "genre_text",
    "tag_text",
    "genome_tag_text"
]

for col in text_columns:
    movies[col] = (
        movies[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    movies["content_text"] = (
    movies["genre_text"] + " " +
    movies["tag_text"] + " " +
    movies["genome_tag_text"]
).str.strip()

    print(
    "Null content_text:",
    movies["content_text"].isna().sum()
)

print(
    "Empty content_text:",
    movies["content_text"].eq("").sum()
)

Null content_text: 0
Null content_text: 0
Null content_text: 0
Empty content_text: 2908


In [70]:

required_columns = [
    "movieId",
    "title",
    "genres",
    "year",
    "clean_title",
    "genre_list",
    "genre_text",
    "tag_text",
    "content_text",
    "genome_tag_text",
    "rating_count",
    "rating_sum",
    "average_rating"
]

assert all(
    col in movies.columns
    for col in required_columns
)

assert movies["movieId"].is_unique

assert (movies["rating_count"] >= 0).all()

assert movies["average_rating"].between(0, 5).all()

assert movies["content_text"].notna().all()

print("\n================================")
print("PHASE 1 VALIDATION PASSED ✅")
print("================================")


print("Null content_text:",
      movies["content_text"].isna().sum())

print(
    movies.loc[
        movies["content_text"].isna(),
        ["movieId", "title", "genres", "genre_text", "tag_text", "genome_tag_text"]
    ].head(10)
)
movies.to_csv(
    "../datasets/processed/movies_phase1_final.csv",
    index=False
)

print("Phase 1 dataset saved.")


PHASE 1 VALIDATION PASSED ✅
Null content_text: 0
Empty DataFrame
Columns: [movieId, title, genres, genre_text, tag_text, genome_tag_text]
Index: []
Phase 1 dataset saved.
